In [ ]:
# =============================
# 1. Imports
# =============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import json
import warnings

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from IPython.display import display

warnings.filterwarnings('ignore')

In [ ]:
# =============================
# 2. Load and prepare data
# =============================
PROJECT_ROOT = Path().resolve().parent
data_path = PROJECT_ROOT / 'data' / 'processed' / 'iberia.csv'

df_pt = pd.read_csv(
    data_path,
    parse_dates=['DateUTC']
)

# Keep relevant columns
df_pt = df_pt[['DateUTC', 'Value']].copy()

# Sort and set index
df_pt = df_pt.sort_values('DateUTC')
df_pt = df_pt.set_index('DateUTC')

# Aggregate duplicates (important)
df_pt = df_pt.groupby(df_pt.index)['Value'].mean().to_frame()

# Ensure hourly frequency
df_pt = df_pt.asfreq('h')

# Fill missing values
df_pt['Value'] = df_pt['Value'].interpolate(method='time')

print('Data ready:', df_pt.shape)

In [ ]:
# =============================
# 3. Fetch exogenous variable
# =============================
# Hourly temperature averaged across 10 cities per country (PT + ES)
# Source: Open-Meteo historical archive API (free, no registration)

def fetch_hourly_temp(cities, start, end):
    temps = {}
    for city, (lat, lon) in cities.items():
        url = (
            f'https://archive-api.open-meteo.com/v1/archive?'
            f'latitude={lat}&longitude={lon}'
            f'&start_date={start}&end_date={end}'
            f'&hourly=temperature_2m'
            f'&timezone=UTC'
        )
        with urllib.request.urlopen(url) as r:
            data = json.loads(r.read())
        temps[city] = pd.Series(
            data['hourly']['temperature_2m'],
            index=pd.to_datetime(data['hourly']['time'])
        )
        print(f'  {city}: OK')
    return pd.DataFrame(temps).mean(axis=1)


PT_CITIES = {
    'Lisboa':  (38.72, -9.14), 'Porto':   (41.15, -8.61),
    'Braga':   (41.55, -8.43), 'Coimbra': (40.21, -8.43),
    'Aveiro':  (40.64, -8.65), 'Setubal': (38.52, -8.89),
    'Evora':   (38.57, -7.91), 'Faro':    (37.02, -7.93),
    'Viseu':   (40.66, -7.91), 'Beja':    (38.02, -7.86),
}

ES_CITIES = {
    'Madrid':     (40.42, -3.70), 'Barcelona':  (41.39,  2.15),
    'Sevilla':    (37.39, -5.99), 'Valencia':   (39.47, -0.38),
    'Bilbao':     (43.26, -2.93), 'Zaragoza':   (41.65, -0.87),
    'Malaga':     (36.72, -4.42), 'Murcia':     (37.98, -1.13),
    'Valladolid': (41.65, -4.72), 'Palma':      (39.57,  2.65),
}

START = '2022-09-30'
END   = '2025-09-30'

print('Portugal...')
temp_pt = fetch_hourly_temp(PT_CITIES, START, END)

print('Spain...')
temp_es = fetch_hourly_temp(ES_CITIES, START, END)

# Average PT + ES to match the load series
temp = pd.DataFrame({'PT': temp_pt, 'ES': temp_es}).mean(axis=1)
temp.name = 'temp'
temp.index = temp.index.tz_localize(None)

print(f'\nTemperature ready: {len(temp)} hours')

In [ ]:
# =============================
# 4. Combine load + exogenous
# =============================
df_pt = df_pt.join(temp, how='inner')

# Calendar features (needed for ARIMAX — no seasonal component)
df_pt['dayofweek'] = df_pt.index.dayofweek
df_pt['month']     = df_pt.index.month
df_pt = df_pt.dropna()

print('Data ready:', df_pt.shape)
print(df_pt.head())

In [ ]:
# =============================
# 5. Plot series
# =============================
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df_pt.index, df_pt['Value'], color='#1f4e79', lw=0.8)
axes[0].set_title('Iberia Electricity Load — PT+ES avg (MW)', fontsize=11)
axes[0].set_ylabel('Load (MW)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_pt.index, df_pt['temp'], color='#e84855', lw=0.8)
axes[1].set_title('Iberia Average Temperature — PT+ES (°C)', fontsize=11)
axes[1].set_ylabel('°C')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Iberia — ARIMAX Variables (2022–2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/arimax_iberia_series.png', dpi=300)
plt.show()

In [ ]:
# =============================
# 6. Stationarity — ADF Test
# =============================
def adf_test(series, name):
    r = adfuller(series, autolag='AIC')
    status = '✓ Stationary' if r[1] < 0.05 else '✗ Non-stationary'
    print(f'ADF [{name}]:')
    print(f'  Statistic : {r[0]:.4f}')
    print(f'  p-value   : {r[1]:.4f}')
    print(f'  Result    : {status}')
    print()

adf_test(df_pt['Value'], 'Load (Iberia avg)')
adf_test(df_pt['temp'],  'Temperature (Iberia avg)')

In [ ]:
# =============================
# 7. Train / Test split
# =============================
df_pt = df_pt.sort_index()

EXOG_COLS = ['temp', 'dayofweek', 'month']

train = df_pt.loc[:'2025-07-31'].copy()
test  = df_pt.loc['2025-08-01':].copy()

print('Train:', train.shape)
print('Test:', test.shape)
print('Test period:', test.index.min(), '→', test.index.max())

In [ ]:
# =============================
# 8. Fit ARIMAX model
# =============================
# ARIMAX(p,d,q) + Xt*beta
# Extension of ARIMA with exogenous variables — no seasonal component
# Same order as 06-arima.ipynb: (2,1,2)
#
# Exogenous variables (Xt):
#   temp       : Iberian Peninsula hourly average temperature (°C)
#   dayofweek  : day of week (0=Monday, 6=Sunday)
#   month      : month of year (1-12)
#
# beta measures the marginal impact of each variable on load (MW)

model_arimax = SARIMAX(
    train['Value'],
    exog=train[EXOG_COLS],
    order=(2, 1, 2),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

fit_arimax = model_arimax.fit(disp=False)

display(fit_arimax.summary())

In [ ]:
# =============================
# 9. ARIMAX diagnostics
# =============================
print('--- Exogenous Coefficients (beta) ---')
for col in EXOG_COLS:
    beta = fit_arimax.params[col]
    pval = fit_arimax.pvalues[col]
    sig  = '✓ significant' if pval < 0.05 else '✗ not significant'
    print(f'  {col:<12}: beta = {beta:>8.3f}   p = {pval:.4f}   {sig}')

print()
print('--- Model Selection Criteria ---')
print(f'  AIC : {fit_arimax.aic:.2f}')
print(f'  BIC : {fit_arimax.bic:.2f}')

print()
print('--- Ljung-Box Test (residuals) ---')
lb = acorr_ljungbox(fit_arimax.resid, lags=12, return_df=True)
print(lb[['lb_stat', 'lb_pvalue']].round(4).to_string())

In [ ]:
# =============================
# 10. 24-hour forecast
# =============================
forecast_24h = fit_arimax.forecast(steps=24, exog=test[EXOG_COLS].iloc[:24])

actual_24h   = test['Value'].iloc[:24]
forecast_24h = pd.Series(forecast_24h.values, index=actual_24h.index)

df_eval_24h = pd.DataFrame({
    'actual':   actual_24h,
    'forecast': forecast_24h
}).dropna()

mae_24  = mean_absolute_error(df_eval_24h['actual'], df_eval_24h['forecast'])
rmse_24 = np.sqrt(mean_squared_error(df_eval_24h['actual'], df_eval_24h['forecast']))
mape_24 = np.mean(np.abs((df_eval_24h['actual'] - df_eval_24h['forecast']) / df_eval_24h['actual'])) * 100

print('\n--- 24h Forecast ---')
print(f'MAE:  {mae_24:.2f}')
print(f'RMSE: {rmse_24:.2f}')
print(f'MAPE: {mape_24:.2f}%')

In [ ]:
# =============================
# 11. Plot 24h forecast
# =============================
plt.figure(figsize=(12, 5))

plt.plot(train.index[-72:], train['Value'].iloc[-72:], label='Train (last 72h)')
plt.plot(actual_24h.index,  actual_24h,                label='Actual (next 24h)')
plt.plot(actual_24h.index,  forecast_24h,              label='ARIMAX Forecast (24h)')

plt.title('Iberia Electricity Load - 24 Hour Ahead ARIMAX Forecast')
plt.xlabel('Date')
plt.ylabel('Load (MW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/arimax_forecast_24h.png', dpi=300)
plt.show()

In [ ]:
# =============================
# 12. Scenario forecasts — 24h
# =============================
# Following the professor's approach (slides 75-76):
# Scenario 1: exogenous variables fixed at last observed values
# Scenario 2: exogenous variables using actual future values

last_obs = train[EXOG_COLS].iloc[-1]

# Scenario 1 — fixed exogenous
exog_s1 = pd.DataFrame(
    [last_obs.values] * 24,
    index=test.index[:24],
    columns=EXOG_COLS
)
s1_obj  = fit_arimax.get_forecast(steps=24, exog=exog_s1)
s1_mean = s1_obj.predicted_mean
s1_ci   = s1_obj.conf_int(alpha=0.05)

# Scenario 2 — actual future exogenous
s2_obj  = fit_arimax.get_forecast(steps=24, exog=test[EXOG_COLS].iloc[:24])
s2_mean = s2_obj.predicted_mean
s2_ci   = s2_obj.conf_int(alpha=0.05)

# Table — Scenario 1
table_s1 = pd.DataFrame({
    'Lim_inf':   s1_ci.iloc[:, 0].round(2),
    'Previsão':  s1_mean.round(2),
    'Lim_sup':   s1_ci.iloc[:, 1].round(2),
    'temp':      exog_s1['temp'].round(2),
    'dayofweek': exog_s1['dayofweek'].astype(int),
    'month':     exog_s1['month'].astype(int),
})
print('--- Cenário 1 (exógenas fixas no último valor observado) ---')
print(table_s1.to_string())

# Table — Scenario 2
table_s2 = pd.DataFrame({
    'Lim_inf':   s2_ci.iloc[:, 0].round(2),
    'Previsão':  s2_mean.round(2),
    'Lim_sup':   s2_ci.iloc[:, 1].round(2),
    'temp':      test['temp'].iloc[:24].round(2),
    'dayofweek': test['dayofweek'].iloc[:24].astype(int),
    'month':     test['month'].iloc[:24].astype(int),
})
print('\n--- Cenário 2 (exógenas com valores futuros reais) ---')
print(table_s2.to_string())

In [ ]:
# =============================
# 13. Plot scenario forecasts
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

context = train['Value'].iloc[-72:]

for ax, (forecast_s, conf_int_s, label) in zip(axes, [
    (s1_mean, s1_ci, 'Cenário 1: exógenas fixas no último valor observado'),
    (s2_mean, s2_ci, 'Cenário 2: exógenas com valores futuros reais'),
]):
    ax.plot(context.index,        context,                 color='#1f4e79', lw=0.8, label='Treino')
    ax.plot(test.index[:24],      test['Value'].iloc[:24], color='#2d6a4f', lw=1.2, label='Teste')
    ax.plot(forecast_s.index,     forecast_s,              color='#e84855', lw=1.2, ls='--', label='ARIMAX')
    ax.fill_between(conf_int_s.index,
                    conf_int_s.iloc[:, 0],
                    conf_int_s.iloc[:, 1],
                    color='#e84855', alpha=0.15)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('Date')
    ax.set_ylabel('Load (MW)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('ARIMAX — Previsão por Cenários', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/arimax_scenarios.png', dpi=300)
plt.show()